# HE Inference: Student with Static/Frozen LayerNorm

Steps 1–2:
1. **Load** student checkpoint (92%+) from Drive; verify `norm_mode="layernorm"` and plaintext test accuracy.
2. **Replace** LayerNorm with Static/Frozen LayerNorm (calibration → affine `y = scale*x + bias`); verify accuracy stays close.

In [ ]:
# Colab setup: clone repo, install deps, device
import sys
from pathlib import Path

REPO_URL = "https://github.com/PulockDas/Secure-Inference-Token-Reduced-VIT.git"
PROJECT_DIR = "/content/Secure-Inference-Token-Reduced-VIT"

if not Path(PROJECT_DIR).exists():
    !git clone $REPO_URL $PROJECT_DIR
%cd $PROJECT_DIR
!git fetch origin
!git checkout feature/he-inference
!git reset --hard origin/feature/he-inference
!pip install -q -r $PROJECT_DIR/requirements.txt

sys.path.insert(0, PROJECT_DIR)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Mount Google Drive (student checkpoint is saved here)
from google.colab import drive
drive.mount("/content/drive")

STUDENT_CKPT = "/content/drive/MyDrive/Secure-Inference-Token-Reduced-VIT/checkpoints/student_best.pt"
if not Path(STUDENT_CKPT).exists():
    print("WARNING: Student checkpoint not found at", STUDENT_CKPT)
    print("Upload student_best.pt to that path, or set STUDENT_CKPT.")
else:
    print("Student checkpoint found:", STUDENT_CKPT)

In [ ]:
# Reload project modules (useful after pulling repo changes)
import sys
for k in list(sys.modules.keys()):
    if k.startswith("models") or k.startswith("training") or k.startswith("data"):
        del sys.modules[k]
!find $PROJECT_DIR -name __pycache__ -type d -exec rm -rf {} + 2>/dev/null; echo "cache cleared"

In [ ]:
# Data: LC25000 train/val/test (same split as training)
from data import get_lc25000_root, get_dataloaders

root = get_lc25000_root()
train_loader, val_loader, test_loader = get_dataloaders(
    root_dir=root, batch_size=32, val_ratio=0.15, test_ratio=0.15, seed=42,
    subdir_depth=2, image_size=224, num_workers=2,
)
ds = train_loader.dataset
num_classes = ds.num_classes
print("Classes:", ds.class_names)
print("Train:", len(ds), "Val:", len(val_loader.dataset), "Test:", len(test_loader.dataset))

## Step 1: Load distilled student and verify plaintext inference

- Load student from Drive with `norm_mode="layernorm"` (must match training).
- Run evaluation on test set and report accuracy (expect 92%+).

In [ ]:
# Load student using shared loader (reads config from checkpoint; strict=False for older ckpts without LayerScale)
from training import load_student_checkpoint

student = load_student_checkpoint(STUDENT_CKPT, device)
student.eval()
print("Student loaded (expect 92%+ plaintext accuracy).")

In [ ]:
# Verify plaintext inference on test set
from training import evaluate_teacher

result_plain = evaluate_teacher(
    student,
    test_loader,
    device,
    ds.class_names,
    results_dir=None,
)
print("Plaintext test accuracy (original LayerNorm):", f"{result_plain['test_acc']:.4f}")
print("Per-class accuracy:", result_plain['per_class_acc'])

## Step 2: Replace LayerNorm with Static/Frozen LayerNorm

- Compute μ_fixed and σ_fixed² on a small calibration set (plaintext).
- Precompute scale and bias so LN becomes affine: y = scale·x + bias.
- Replace all LayerNorm layers with this static version.
- Verify plaintext accuracy remains close to original.

In [ ]:
# Calibration: use a subset of training data (same seed/split as training)
from torch.utils.data import Subset

NUM_CALIBRATION_BATCHES = 16
calib_size = min(NUM_CALIBRATION_BATCHES * train_loader.batch_size, len(train_loader.dataset))
calib_indices = list(range(calib_size))
calib_dataset = Subset(train_loader.dataset, calib_indices)
calib_loader = torch.utils.data.DataLoader(
    calib_dataset,
    batch_size=train_loader.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)
print("Calibration batches:", NUM_CALIBRATION_BATCHES, "samples:", len(calib_dataset))

In [ ]:
# Replace all nn.LayerNorm with StaticLayerNorm (calibration runs inside)
from models import replace_layernorm_with_static

replaced = replace_layernorm_with_static(
    student,
    calib_loader,
    device,
    num_calibration_batches=NUM_CALIBRATION_BATCHES,
    eps=1e-6,
)
print("Replaced LayerNorm modules:", replaced)

In [ ]:
# Verify plaintext accuracy after replacing with StaticLayerNorm
result_static = evaluate_teacher(
    student,
    test_loader,
    device,
    ds.class_names,
    results_dir=None,
)
print("Plaintext test accuracy (StaticLayerNorm):", f"{result_static['test_acc']:.4f}")
print("Per-class accuracy:", result_static['per_class_acc'])
print("Drop from original:", f"{result_plain['test_acc'] - result_static['test_acc']:.4f}")